# PRISM Clinical Confidence Layer
## Unsupervised Discovery of High-Benefit Patient Archetypes

This notebook clusters the top-benefit patients (by DR-learner benefit score) into clinical archetypes, then scores new patients against those archetypes with a **confidence metric** that says how much a given assignment should be trusted.

**Methodology summary:**

1. Cluster in a **PCA-reduced space** (not raw features) — with ~140 high-benefit training patients, clustering directly in 12 raw dimensions overfits badly.
2. Feature set is SHAP-ranked *and* correlation-pruned, so redundant utilization/cost/risk signals don't eat up dimensions without adding separating structure.
3. Primary clustering is a **Gaussian Mixture Model** (`covariance_type="diag"`) — chosen over K-Means because it (a) allows each archetype its own spread instead of assuming spherical, equal-size clusters, and (b) gives soft, probabilistic membership instead of a hard label.
4. Confidence for a new patient is built from **two orthogonal signals**, combined so that failing *either* one drags the score down (not averaged away by the other):
   - **Max posterior probability** — how unambiguous the winning archetype is
   - **Log-likelihood typicality** — whether the patient resembles the training population *at all*, independent of which archetype wins
5. **HDBSCAN** is used as an independent, density-based cross-check on which patients look like outliers — a second, structurally different method used to corroborate (or challenge) the GMM-based outlier calls.
6. Every stability/validity claim is bootstrap-checked (100 resamples) before being trusted.

*(This supersedes prior drafts. It folds in: PCA before clustering, `diag` covariance, correlation pruning, capped k-search, and now — geometric-mean confidence scoring on posterior × typicality, plus HDBSCAN outlier corroboration.)*

---
## Setup
---

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import percentileofscore

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Prefer the standalone hdbscan package (has approximate_predict + prediction_data).
# Fall back to sklearn's HDBSCAN (>=1.3) when the package isn't installed.
try:
    from hdbscan import HDBSCAN, approximate_predict
    _HDBSCAN_BACKEND = 'hdbscan-pkg'
except ImportError:
    from sklearn.cluster import HDBSCAN
    _HDBSCAN_BACKEND = 'sklearn'

for candidate in [Path.cwd(), Path.cwd() / 'Code']:
    if (candidate / '_prism_model_utils.py').exists():
        sys.path.insert(0, str(candidate))
        break

from _prism_model_utils import (
    project_root,
    ensure_output_folder,
    split_train_test,
    ntile_desc,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

PROJECT_ROOT = project_root()
SEED = 123
np.random.seed(SEED)

print('Project root:', PROJECT_ROOT)
print('HDBSCAN backend:', _HDBSCAN_BACKEND)

---
## Step 1 – Load DR-Learner Outputs

Load scored outputs from the Doubly Robust notebook and reconstruct the same deterministic train/test split (seed=123, 70/30, stratified on intervention flag and outcome).
---

In [2]:
DR_OUTPUT_DIR = PROJECT_ROOT / 'Outputs' / 'Doubly-Robust' / 'Python'
OUTPUT_DIR = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Clinical-Confidence-Layer' / 'Python')

TRAIN_FRACTION = 0.70
HIGH_BENEFIT_PERCENTILE = 0.20      # top 20% by benefit score
N_SHAP_FEATURES = 12                # target feature count after correlation pruning
CORR_PRUNE_THRESHOLD = 0.80         # drop features correlated above this with an already-kept feature
PCA_VARIANCE_TARGET = 0.90          # PCA variance retained before clustering
GMM_COVARIANCE_TYPE = 'diag'        # per-archetype variances, not full covariance (n too small for 'full')
GMM_REG_COVAR = 1e-3                # avoid near-singular / overconfident covariances
K_RANGE = range(2, 6)               # capped given n~140 post-PCA
N_BOOTSTRAP = 100                   # bootstrap resamples for stability
KNN_K = 10                          # neighbors for diagnostic k-NN similarity
OUTLIER_PERCENTILE = 0.05           # bottom 5% of train log-likelihood = 'doesn't look like training pop'

scored_full = pd.read_csv(DR_OUTPUT_DIR / 'doubly_robust_scored_output.csv')
scored_test = pd.read_csv(DR_OUTPUT_DIR / 'doubly_robust_scored_test_output.csv')
shap_importance = pd.read_csv(DR_OUTPUT_DIR / 'doubly_robust_global_benefit_shap_importance.csv')

print(f'Full scored population: {len(scored_full)} members')
print(f'Test scored population: {len(scored_test)} members')
print(f'SHAP features available: {len(shap_importance)}')

train_df, test_df = split_train_test(
    scored_full,
    train_fraction=TRAIN_FRACTION,
    seed=SEED,
    stratify_columns=['intervention_flag', 'outcome_ed_90d'],
)

print(f'\nReconstructed split: train={len(train_df)}, test={len(test_df)}')

---
## Step 2 – Construct High-Benefit Reference Population

Select the top 20% of members by predicted benefit score, separately in train and test.
---

In [3]:
benefit_threshold_train = train_df['benefit_score'].quantile(1 - HIGH_BENEFIT_PERCENTILE)
high_benefit_train = train_df[train_df['benefit_score'] >= benefit_threshold_train].copy()

benefit_threshold_test = test_df['benefit_score'].quantile(1 - HIGH_BENEFIT_PERCENTILE)
high_benefit_test = test_df[test_df['benefit_score'] >= benefit_threshold_test].copy()

print(f'High-benefit training population: {len(high_benefit_train)} (threshold >= {benefit_threshold_train:.4f})')
print(f'High-benefit test population: {len(high_benefit_test)} (threshold >= {benefit_threshold_test:.4f})')

# Export data review summary for README generator
pd.DataFrame({'metric': [
    'Full scored population', 'Reconstructed train members', 'Reconstructed test members',
    'High-benefit train members (top 20%)', 'High-benefit test members (top 20%)',
    'High-benefit train benefit-score threshold', 'High-benefit test benefit-score threshold',
], 'value': [
    f'{len(scored_full):,}', f'{len(train_df):,}', f'{len(test_df):,}',
    f'{len(high_benefit_train)}', f'{len(high_benefit_test)}',
    f'>= {benefit_threshold_train:.4f}', f'>= {benefit_threshold_test:.4f}',
]}).to_csv(OUTPUT_DIR / 'clinical_confidence_data_review_summary.csv', index=False)

---
## Step 3 – Feature Selection: SHAP Ranking + Correlation Pruning

Walk the SHAP-ranked feature list in order, greedily skipping any feature whose absolute correlation with an already-kept feature exceeds `CORR_PRUNE_THRESHOLD`. This prevents near-duplicate signals (e.g. `current_risk_score`, `percolator_utilization_score`, `total_cost_last_6m`, `ed_visits_last_6m` all measuring 'how sick/costly') from consuming multiple feature slots without adding real separating structure.
---

In [4]:
candidate_pool = (
    shap_importance
    .nlargest(min(len(shap_importance), N_SHAP_FEATURES * 3), 'mean_abs_benefit_shap')['feature']
    .tolist()
)
candidate_pool = [f for f in candidate_pool if f in high_benefit_train.columns]

candidate_matrix = high_benefit_train[candidate_pool].apply(pd.to_numeric, errors='coerce').fillna(0)
corr_matrix = candidate_matrix.corr().abs()

kept_features, dropped_features = [], []
for feat in candidate_pool:
    if not kept_features:
        kept_features.append(feat)
        continue
    max_corr_with_kept = corr_matrix.loc[feat, kept_features].max()
    if max_corr_with_kept >= CORR_PRUNE_THRESHOLD:
        dropped_features.append((feat, max_corr_with_kept))
    else:
        kept_features.append(feat)
    if len(kept_features) >= N_SHAP_FEATURES:
        break

CLUSTER_FEATURES = kept_features[:N_SHAP_FEATURES]

print(f'Selected {len(CLUSTER_FEATURES)} clustering features after correlation pruning:')
for i, feat in enumerate(CLUSTER_FEATURES, 1):
    print(f'  {i:2d}. {feat}')
if dropped_features:
    print(f'\nDropped as redundant (corr >= {CORR_PRUNE_THRESHOLD}):')
    for feat, c in dropped_features:
        print(f'  {feat} (corr={c:.2f})')

X_train_raw = high_benefit_train[CLUSTER_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_raw = high_benefit_test[CLUSTER_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)   # fit ONLY on train — no leakage
X_test_scaled = scaler.transform(X_test_raw)

print(f'\nScaled shapes: train={X_train_scaled.shape}, test={X_test_scaled.shape}')

---
## Step 3b – Dimensionality Reduction (PCA)

With only ~140 training patients, clustering directly in 12-D asks a covariance-based model to fit far more parameters than there are data points. Reduce to a PCA space retaining `PCA_VARIANCE_TARGET` of the variance and cluster there; archetypes are still profiled back on the original clinical features afterward.
---

In [5]:
pca_cluster = PCA(n_components=PCA_VARIANCE_TARGET, random_state=SEED)
X_train_pca = pca_cluster.fit_transform(X_train_scaled)
X_test_pca = pca_cluster.transform(X_test_scaled)

print(f'PCA components retained: {pca_cluster.n_components_} (target variance {PCA_VARIANCE_TARGET:.0%})')
print(f'Variance explained per component: {np.round(pca_cluster.explained_variance_ratio_, 3)}')
print(f'Cumulative variance explained: {pca_cluster.explained_variance_ratio_.sum():.1%}')
print(f'\nClustering matrix shapes: train={X_train_pca.shape}, test={X_test_pca.shape}')

# Export PCA variance for README generator
pd.DataFrame({
    'component': [f'PC{i+1}' for i in range(pca_cluster.n_components_)],
    'variance_explained': pca_cluster.explained_variance_ratio_,
}).to_csv(OUTPUT_DIR / 'clinical_confidence_pca_variance.csv', index=False)

---
## Step 4 – Discover High-Benefit Patient Archetypes

**Primary model:** Gaussian Mixture Model, BIC-selected `k`, `covariance_type="diag"`.

**Why GMM over K-Means:** K-Means assumes spherical, equal-size clusters and only produces hard assignments. Real clinical archetypes rarely have uniform spread — one archetype may be tight, another diffuse — and boundary patients deserve a soft, probabilistic membership rather than a forced single label. GMM's per-component covariance and posterior probabilities give both directly.

**Cross-checks:** K-Means and Agglomerative Clustering (via ARI agreement with GMM).
**Density diagnostic:** HDBSCAN — fit here on the training set so it's available later both as a stability cross-check and as an independent outlier-verification method (Step 7b).
---

In [6]:
# --- GMM model selection via BIC/AIC ---
bic_scores, aic_scores = [], []
for k in K_RANGE:
    candidate = GaussianMixture(
        n_components=k, covariance_type=GMM_COVARIANCE_TYPE,
        reg_covar=GMM_REG_COVAR, random_state=SEED, n_init=10,
    ).fit(X_train_pca)
    bic_scores.append(candidate.bic(X_train_pca))
    aic_scores.append(candidate.aic(X_train_pca))

optimal_k = list(K_RANGE)[np.argmin(bic_scores)]
print(f'BIC-optimal number of archetypes: {optimal_k}')
print(f'BIC by k: {dict(zip(K_RANGE, np.round(bic_scores, 1)))}')

# Export BIC-by-k for README generator
pd.DataFrame({'k': list(K_RANGE), 'bic': bic_scores, 'aic': aic_scores}).to_csv(
    OUTPUT_DIR / 'clinical_confidence_bic_by_k.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(K_RANGE, bic_scores, 'o-', linewidth=2, label='BIC')
ax.plot(K_RANGE, aic_scores, 's--', linewidth=2, label='AIC')
ax.axvline(optimal_k, color='red', linestyle=':', label=f'Optimal k={optimal_k}')
ax.set_xlabel('Number of archetypes (k)'); ax.set_ylabel('Information criterion')
ax.set_title('GMM Model Selection (PCA space)'); ax.legend()
plt.tight_layout(); fig.savefig(OUTPUT_DIR / 'gmm_bic_aic_selection.png', dpi=150); plt.show()

# --- Fit final models ---
gmm = GaussianMixture(
    n_components=optimal_k, covariance_type=GMM_COVARIANCE_TYPE,
    reg_covar=GMM_REG_COVAR, random_state=SEED, n_init=10,
)
gmm_labels = gmm.fit_predict(X_train_pca)

kmeans = KMeans(n_clusters=optimal_k, random_state=SEED, n_init=20)
kmeans_labels = kmeans.fit_predict(X_train_pca)

agg = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
agg_labels = agg.fit_predict(X_train_pca)

# HDBSCAN fit on the training set — used for (a) a rough independent cluster-count / noise
# check now, and (b) outlier corroboration on test patients in Step 7b.
HDBSCAN_MIN_CLUSTER_SIZE = max(5, len(X_train_pca) // 20)
_hdb_kwargs = dict(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=3)
if _HDBSCAN_BACKEND == 'hdbscan-pkg':
    _hdb_kwargs['prediction_data'] = True  # only the hdbscan package supports this
hdbscan_model = HDBSCAN(**_hdb_kwargs)
hdbscan_train_labels = hdbscan_model.fit_predict(X_train_pca)

print('\nCluster sizes:')
print('  GMM:          ', dict(zip(*np.unique(gmm_labels, return_counts=True))))
print('  K-Means:      ', dict(zip(*np.unique(kmeans_labels, return_counts=True))))
print('  Agglomerative:', dict(zip(*np.unique(agg_labels, return_counts=True))))

train_noise = (hdbscan_train_labels == -1)
print(f'  HDBSCAN:       {len(set(hdbscan_train_labels) - {-1})} clusters, '
      f'{train_noise.sum()} noise points ({train_noise.mean():.1%})')

# --- Agreement across methods ---
ari_gmm_kmeans = adjusted_rand_score(gmm_labels, kmeans_labels)
ari_gmm_agg = adjusted_rand_score(gmm_labels, agg_labels)
sil_gmm = silhouette_score(X_train_pca, gmm_labels)

# Extended validation metrics for all methods (silhouette only)
sil_kmeans = silhouette_score(X_train_pca, kmeans_labels)
sil_agg = silhouette_score(X_train_pca, agg_labels)

print(f'\nSilhouette (GMM, PCA space): {sil_gmm:.3f}')
print(f'ARI GMM vs K-Means: {ari_gmm_kmeans:.3f}')
print(f'ARI GMM vs Agglomerative: {ari_gmm_agg:.3f}')

# --- HDBSCAN: real per-cluster sizes (+noise) and validation on non-noise points only ---
import collections as _collections
_hdb_counts = _collections.Counter(hdbscan_train_labels)
hdb_sizes = '/'.join(str(_hdb_counts[k]) for k in sorted(k for k in _hdb_counts if k != -1))
hdb_sizes = f'{hdb_sizes} (+{_hdb_counts[-1]} noise)' if hdb_sizes else f'0 (+{_hdb_counts[-1]} noise)'

_nn = ~train_noise  # non-noise mask
if _nn.sum() > optimal_k and len(set(hdbscan_train_labels[_nn])) > 1:
    sil_hdb = silhouette_score(X_train_pca[_nn], hdbscan_train_labels[_nn])
else:
    sil_hdb = np.nan

print(f'HDBSCAN internal validation (non-noise only, n={int(_nn.sum())}): '
      f'silhouette={sil_hdb:.3f}')

# Export clustering method comparison CSV for README generator
gmm_sizes = '/'.join(f'{v}' for v in np.unique(gmm_labels, return_counts=True)[1])
km_sizes = '/'.join(f'{v}' for v in np.unique(kmeans_labels, return_counts=True)[1])
agg_sizes = '/'.join(f'{v}' for v in np.unique(agg_labels, return_counts=True)[1])
hdb_n_clusters = len(set(hdbscan_train_labels) - {-1})
pd.DataFrame([
    {'method': 'Gaussian Mixture', 'n_clusters': optimal_k, 'cluster_sizes': gmm_sizes,
     'noise_points': 0, 'silhouette': sil_gmm},
    {'method': 'K-Means', 'n_clusters': optimal_k, 'cluster_sizes': km_sizes,
     'noise_points': 0, 'silhouette': sil_kmeans},
    {'method': 'Agglomerative', 'n_clusters': optimal_k, 'cluster_sizes': agg_sizes,
     'noise_points': 0, 'silhouette': sil_agg},
    {'method': f'HDBSCAN (non-noise, n={int(_nn.sum())})', 'n_clusters': hdb_n_clusters, 'cluster_sizes': hdb_sizes,
     'noise_points': int(train_noise.sum()), 'silhouette': sil_hdb},
]).to_csv(OUTPUT_DIR / 'clinical_confidence_clustering_method_comparison.csv', index=False)

### Archetype Profiles

Summarize each archetype on the original clinical features.

In [7]:
high_benefit_train_labeled = high_benefit_train.copy()
high_benefit_train_labeled['archetype'] = gmm_labels

profile_features = CLUSTER_FEATURES + ['benefit_score']
archetype_summary_rows = []
for cluster_id in sorted(high_benefit_train_labeled['archetype'].unique()):
    cluster_data = high_benefit_train_labeled[high_benefit_train_labeled['archetype'] == cluster_id]
    row = {'archetype': cluster_id, 'n_patients': len(cluster_data),
           'avg_benefit_score': cluster_data['benefit_score'].mean()}
    for feat in CLUSTER_FEATURES:
        row[f'avg_{feat}'] = cluster_data[feat].mean()
    archetype_summary_rows.append(row)

archetype_summary = pd.DataFrame(archetype_summary_rows)
archetype_summary.to_csv(OUTPUT_DIR / 'archetype_summary.csv', index=False)
display(archetype_summary)

# ============================================================================
# Flag ablation: is the GMM split substantially a re-encoding of the two flags?
# ============================================================================
_flag_cols = [c for c in ['anxiety_flag', 'copd_flag'] if c in high_benefit_train.columns]
if len(_flag_cols) == 2:
    anx = pd.to_numeric(high_benefit_train['anxiety_flag'], errors='coerce').fillna(0).to_numpy()
    cpd = pd.to_numeric(high_benefit_train['copd_flag'], errors='coerce').fillna(0).to_numpy()
    either = ((anx + cpd) > 0).astype(int)
    ari_anx = adjusted_rand_score(gmm_labels, anx)
    ari_cpd = adjusted_rand_score(gmm_labels, cpd)
    ari_either = adjusted_rand_score(gmm_labels, either)

    # Refit GMM on continuous-only features (flags removed)
    cont_feats = [c for c in CLUSTER_FEATURES if c not in ('anxiety_flag', 'copd_flag')]
    Xc_scaled = StandardScaler().fit_transform(
        high_benefit_train[cont_feats].apply(pd.to_numeric, errors='coerce').fillna(0))
    Xc_pca = PCA(n_components=PCA_VARIANCE_TARGET, random_state=SEED).fit_transform(Xc_scaled)
    gmm_cont = GaussianMixture(n_components=optimal_k, covariance_type=GMM_COVARIANCE_TYPE,
                               reg_covar=GMM_REG_COVAR, random_state=SEED, n_init=10).fit(Xc_pca)
    cont_labels = gmm_cont.predict(Xc_pca)
    ari_orig_vs_cont = adjusted_rand_score(gmm_labels, cont_labels)
    sil_cont = silhouette_score(Xc_pca, cont_labels)

    flag_ablation = pd.DataFrame([
        {'comparison': 'GMM labels vs anxiety_flag', 'value': ari_anx},
        {'comparison': 'GMM labels vs copd_flag', 'value': ari_cpd},
        {'comparison': 'GMM labels vs (anxiety OR copd)', 'value': ari_either},
        {'comparison': 'Original vs continuous-only GMM (ARI)', 'value': ari_orig_vs_cont},
        {'comparison': 'Silhouette (continuous-only fit)', 'value': sil_cont},
    ])
    flag_ablation.to_csv(OUTPUT_DIR / 'clinical_confidence_flag_ablation.csv', index=False)

    print('\nFlag ablation (ARI unless noted):')
    display(flag_ablation)

# ============================================================================
# Archetype benefit-difference test: do archetypes actually stratify benefit?
# ============================================================================
if optimal_k == 2:
    bs = high_benefit_train['benefit_score'].to_numpy()
    a = bs[gmm_labels == 0]
    b = bs[gmm_labels == 1]
    diff = float(b.mean() - a.mean())
    se = float(np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)))
    benefit_diff_test = pd.DataFrame([
        {'metric': 'mean_benefit_archetype_0', 'value': float(a.mean())},
        {'metric': 'mean_benefit_archetype_1', 'value': float(b.mean())},
        {'metric': 'benefit_difference', 'value': diff},
        {'metric': 'ci95_lower', 'value': diff - 1.96 * se},
        {'metric': 'ci95_upper', 'value': diff + 1.96 * se},
    ])
    benefit_diff_test.to_csv(OUTPUT_DIR / 'clinical_confidence_benefit_difference_test.csv', index=False)
    ci_spans_zero = (diff - 1.96 * se) <= 0 <= (diff + 1.96 * se)
    print(f'\nArchetype benefit difference: {diff:.4f} '
          f'(95% CI {diff - 1.96 * se:.4f} to {diff + 1.96 * se:.4f}); '
          f'CI spans zero: {ci_spans_zero}')

---
## Step 5 – Validate Cluster Stability

Stability gates whether these archetypes should be presented as reproducible clinical phenotypes at all. Bootstrap resample the training set, refit GMM each time, and measure agreement (ARI) against the original labels.
---

In [8]:
print('Internal validation (GMM, PCA space):')
print(f'  Silhouette:       {sil_gmm:.3f}  (higher better, range [-1,1])')

rng = np.random.default_rng(SEED)
bootstrap_ari_scores = []
for i in range(N_BOOTSTRAP):
    idx = rng.choice(len(X_train_pca), size=len(X_train_pca), replace=True)
    gmm_boot = GaussianMixture(
        n_components=optimal_k, covariance_type=GMM_COVARIANCE_TYPE,
        reg_covar=GMM_REG_COVAR, random_state=SEED + i, n_init=5,
    ).fit(X_train_pca[idx])
    boot_labels = gmm_boot.predict(X_train_pca)
    bootstrap_ari_scores.append(adjusted_rand_score(gmm_labels, boot_labels))

bootstrap_ari = np.array(bootstrap_ari_scores)
# Bands: STRONG >= 0.75 | MODERATE 0.50-0.75 | WEAK < 0.50
stability_assessment = ('STRONG' if bootstrap_ari.mean() >= 0.75
                        else 'MODERATE' if bootstrap_ari.mean() >= 0.50 else 'WEAK')

print(f'\nBootstrap stability ({N_BOOTSTRAP} iters): mean ARI={bootstrap_ari.mean():.3f}, '
      f'std={bootstrap_ari.std():.3f}  -> {stability_assessment}')

# Persist the full array so the distribution can be rendered
np.savetxt(OUTPUT_DIR / 'bootstrap_ari_values.csv', bootstrap_ari, delimiter=',')

# Histogram of bootstrap ARI with band thresholds
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(bootstrap_ari, bins=20, color='#4C78A8', edgecolor='white')
ax.axvline(bootstrap_ari.mean(), color='black', linestyle='-', linewidth=2, label=f'mean={bootstrap_ari.mean():.3f}')
ax.axvline(0.50, color='orange', linestyle='--', label='MODERATE >= 0.50')
ax.axvline(0.75, color='green', linestyle='--', label='STRONG >= 0.75')
ax.set_xlabel('Adjusted Rand Index vs. reference labeling')
ax.set_ylabel('Bootstrap resamples')
ax.set_title(f'Bootstrap Stability Distribution ({N_BOOTSTRAP} resamples) — {stability_assessment}')
ax.legend()
plt.tight_layout(); fig.savefig(OUTPUT_DIR / 'bootstrap_ari_distribution.png', dpi=150); plt.show()

stability_summary = pd.DataFrame([{
    'n_components': optimal_k, 'silhouette_score': sil_gmm,
    'bootstrap_mean_ari': bootstrap_ari.mean(),
    'bootstrap_std_ari': bootstrap_ari.std(), 'bootstrap_min_ari': bootstrap_ari.min(),
    'ari_gmm_vs_kmeans': ari_gmm_kmeans,
    'ari_gmm_vs_agglomerative': ari_gmm_agg, 'stability_assessment': stability_assessment,
}])
stability_summary.to_csv(OUTPUT_DIR / 'cluster_stability_summary.csv', index=False)
display(stability_summary)

---
## Step 6 – Score Test Patients: Posterior, Typicality, and Diagnostics

Two signals answer two different questions, and both are needed:

- **`gmm_max_posterior`** — *given* that a patient belongs to some archetype, how unambiguous is the winner? (High when a patient clearly resembles one archetype far more than any other.)
- **`typicality_score`** — does the patient resemble the training population *at all*? Computed from GMM log-likelihood (`score_samples`), converted to a percentile rank against the training distribution so it's on a comparable [0,1] scale to the posterior.

A patient can have a very high posterior while still being far from every archetype — GMM always confidently assigns to *someone*, even a true outlier. Typicality is what catches that.

`knn_similarity` is also computed as an independent diagnostic (not part of the confidence score) — a simple, model-free way to sanity-check the GMM-based signals.
---

In [9]:
# --- GMM posterior (relative ambiguity between archetypes) ---
test_gmm_proba = gmm.predict_proba(X_test_pca)
test_gmm_max_posterior = test_gmm_proba.max(axis=1)
test_gmm_labels = gmm.predict(X_test_pca)

# --- Log-likelihood typicality (absolute resemblance to training population) ---
train_loglik = gmm.score_samples(X_train_pca)
test_loglik = gmm.score_samples(X_test_pca)

typicality_score = np.array([
    percentileofscore(train_loglik, val) / 100.0
    for val in test_loglik
])

is_archetype_outlier = typicality_score < OUTLIER_PERCENTILE

# --- k-NN similarity (independent diagnostic, not folded into the confidence score) ---
knn = NearestNeighbors(n_neighbors=KNN_K, metric='euclidean').fit(X_train_pca)
distances, _ = knn.kneighbors(X_test_pca)
knn_similarity = 1.0 / (1.0 + distances.mean(axis=1))

print('GMM posterior:      mean={:.3f}, std={:.3f}'.format(test_gmm_max_posterior.mean(), test_gmm_max_posterior.std()))
print('Typicality score:   mean={:.3f}, std={:.3f}'.format(typicality_score.mean(), typicality_score.std()))
print('k-NN similarity:    mean={:.3f}, std={:.3f}'.format(knn_similarity.mean(), knn_similarity.std()))
print(f'\nFlagged as archetype outliers (typicality < {OUTLIER_PERCENTILE:.0%} of train): '
      f'{is_archetype_outlier.sum()} / {len(is_archetype_outlier)} test patients')

# --- Degeneracy check: a component with ~zero variance contributes no discriminating signal ---
DEGENERACY_STD_THRESHOLD = 0.01
for name, arr in [('gmm_posterior', test_gmm_max_posterior), ('typicality', typicality_score),
                  ('knn_similarity', knn_similarity)]:
    flag = '  <-- WARNING: near-zero variance, no discriminating signal' if arr.std() < DEGENERACY_STD_THRESHOLD else ''
    print(f'{name}: std={arr.std():.4f}{flag}')

# --- Generalization / overfitting signal: KS test on train vs test log-likelihoods ---
# If the high-benefit train and test populations were exchangeable under the fitted mixture,
# mean test typicality would sit near 0.50. Below that => density fit partly to training idiosyncrasy.
from scipy import stats as _spstats_ks
ks_stat, ks_p = _spstats_ks.ks_2samp(train_loglik, test_loglik)
print(f'\nTypicality generalization: mean test typicality={typicality_score.mean():.3f} '
      f'(0.50 expected under exchangeability)')
print(f'KS(train loglik vs test loglik): stat={ks_stat:.3f}, p={ks_p:.4f}; '
      f'median train={np.median(train_loglik):.2f}, median test={np.median(test_loglik):.2f}')
pd.DataFrame([
    {'metric': 'mean_test_typicality', 'value': float(typicality_score.mean())},
    {'metric': 'expected_under_exchangeability', 'value': 0.50},
    {'metric': 'ks_statistic', 'value': float(ks_stat)},
    {'metric': 'ks_pvalue', 'value': float(ks_p)},
    {'metric': 'median_train_loglik', 'value': float(np.median(train_loglik))},
    {'metric': 'median_test_loglik', 'value': float(np.median(test_loglik))},
]).to_csv(OUTPUT_DIR / 'clinical_confidence_typicality_generalization.csv', index=False)

---
## Step 7 – Combined Confidence Score (Penalizes Failing Either Axis)

Posterior and typicality are combined with a **geometric mean**, not an arithmetic average. This matters concretely: a patient with posterior=0.95 but typicality=0.05 (confidently assigned, but doesn't look like the training population) would score ~0.50 under an arithmetic mean — a misleading 'Medium.' Under the geometric mean it scores ~0.22 — correctly 'Low,' because trustworthy archetype membership requires **both** conditions to hold, not one compensating for the other.

The separately-computed KDE density term from earlier drafts is retired here — `gmm.score_samples()` measures the same underlying idea (density under the fitted model) but stays internally consistent with the archetypes themselves, and avoids the bandwidth-collapse problem a standalone KDE hit at high dimensionality.
---

In [10]:
# Raw geometric-mean score (test) BEFORE normalization
raw_test = np.sqrt(test_gmm_max_posterior * typicality_score)

# Freeze min-max bounds on the TRAINING high-benefit population so the score is
# comparable across future scoring batches (min-max per-batch would rescale every run,
# making 0/1 trivially true and tier cuts non-transferable).
train_post = gmm.predict_proba(X_train_pca).max(axis=1)
train_typ = np.array([percentileofscore(train_loglik, v) / 100.0 for v in train_loglik])
raw_train = np.sqrt(train_post * train_typ)
score_lo, score_hi = float(raw_train.min()), float(raw_train.max())
score_range = max(score_hi - score_lo, 1e-10)

confidence_score_normalized = np.clip((raw_test - score_lo) / score_range, 0, 1)

# Report the RAW score distribution (not the post-normalization 0/1, which is circular)
print('Raw sqrt(posterior * typicality) on test — the non-circular summary:')
print(f'  min={raw_test.min():.4f}  max={raw_test.max():.4f}  '
      f'mean={raw_test.mean():.4f}  std={raw_test.std():.4f}')
print(f'\nTraining-derived normalization bounds (persisted): lo={score_lo:.4f}, hi={score_hi:.4f}')
print('Normalized score (fixed train bounds):')
print(f'  mean={confidence_score_normalized.mean():.3f}  std={confidence_score_normalized.std():.3f}  '
      f'range=[{confidence_score_normalized.min():.3f}, {confidence_score_normalized.max():.3f}]')

# Persist raw summary + bounds for the README generator
pd.DataFrame([
    {'metric': 'raw_min', 'value': float(raw_test.min())},
    {'metric': 'raw_max', 'value': float(raw_test.max())},
    {'metric': 'raw_mean', 'value': float(raw_test.mean())},
    {'metric': 'raw_std', 'value': float(raw_test.std())},
    {'metric': 'train_bound_lo', 'value': score_lo},
    {'metric': 'train_bound_hi', 'value': score_hi},
    {'metric': 'normalized_mean', 'value': float(confidence_score_normalized.mean())},
    {'metric': 'normalized_std', 'value': float(confidence_score_normalized.std())},
]).to_csv(OUTPUT_DIR / 'clinical_confidence_combined_score_summary.csv', index=False)

---
## Step 7b – HDBSCAN Cross-Verification of Low-Confidence / Outlier Points

HDBSCAN is density-based and makes none of GMM's Gaussian-shape assumptions — a structurally different method. Rather than using it as the primary clustering approach (it collapsed to ~100% noise on raw 12-D test data in earlier drafts, and gives only a binary in/out call rather than a graded confidence), it's used here purely to **corroborate** which patients the GMM-based approach flags as outliers. Agreement between two different algorithms is much stronger evidence of genuine atypicality than either method alone.
---

In [11]:
# --- Step 7b: Assign test points to train-fitted HDBSCAN clusters (noise = -1) ---

if _HDBSCAN_BACKEND == "hdbscan-pkg":
    # Make sure prediction data exists (safe even if already generated)
    if not hasattr(hdbscan_model, "prediction_data_"):
        hdbscan_model.generate_prediction_data()

    # Predict cluster membership for the test set
    test_hdbscan_labels, test_hdbscan_strengths = approximate_predict(
        hdbscan_model,
        X_test_pca
    )

else:
    # sklearn's HDBSCAN has no approximate_predict().
    # Refit on the combined data and assign labels to the test rows.
    combined_pca = np.vstack([X_train_pca, X_test_pca])

    combined_model = HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=3
    )

    combined_labels = combined_model.fit_predict(combined_pca)

    # Extract labels for the test portion
    test_hdbscan_labels = combined_labels[len(X_train_pca):]
    test_hdbscan_strengths = np.ones(len(test_hdbscan_labels))

# Binary noise flag (1 = noise/outlier, 0 = assigned to a cluster)
test_noise_flag = (test_hdbscan_labels == -1).astype(int)

print(f"Test points assigned to clusters: {(test_noise_flag == 0).sum():,}")
print(f"Test points labeled as noise:    {(test_noise_flag == 1).sum():,}")

---
## Step 8 – Assign Confidence Tiers (Natural Breaks via 1-D GMM)

Rather than cutting at fixed terciles (an arbitrary 33/33/33 split that can slice a genuinely single group in two just to force even thirds), tier boundaries are found by fitting a **1-D Gaussian Mixture Model directly on `confidence_score_normalized`**. This is the same model family used for the archetypes themselves, so the tiering logic stays consistent with the rest of the notebook, and it produces real cut points — wherever the mixture components' densities actually cross — instead of an artificial percentile rule. If the test population's confidence scores naturally split 60/25/15 instead of evenly, that's what comes out.

BIC is used to choose between 2 and 3 natural groups: if the data genuinely only supports two bands, forcing a third would be the same mistake as forcing even terciles. Components are then ordered by their mean confidence (low → high) and mapped to Low/Medium/High (or just Low/High if only 2 groups are found) — the labels are assigned by ordering, not by GMM's arbitrary internal component index.

The **hard override** is unchanged: any patient flagged as an archetype outlier (Step 6) is forced into the Low tier regardless of where the continuous score — or the natural-break boundary — puts them. An outlier flag is a statement that the patient doesn't resemble the training population at all, which should never be quietly outvoted by a middling composite score.
---

In [12]:
# --- Fit a 1-D GMM on the confidence score to find natural breakpoints ---
TIER_K_RANGE = range(2, 4)  # let BIC choose 2 or 3 natural bands, don't assume 3

confidence_col = confidence_score_normalized.reshape(-1, 1)

tier_bic = []
for k in TIER_K_RANGE:
    tier_candidate = GaussianMixture(n_components=k, random_state=SEED, n_init=10)
    tier_candidate.fit(confidence_col)
    tier_bic.append(tier_candidate.bic(confidence_col))

optimal_tier_k = list(TIER_K_RANGE)[np.argmin(tier_bic)]
print(f'BIC by tier count: {dict(zip(TIER_K_RANGE, np.round(tier_bic, 1)))}')
print(f'Natural tier count selected: {optimal_tier_k}')

# Export tier BIC for README generator
pd.DataFrame({'n_tiers': list(TIER_K_RANGE), 'bic': tier_bic}).to_csv(
    OUTPUT_DIR / 'clinical_confidence_tier_bic.csv', index=False)

tier_gmm = GaussianMixture(n_components=optimal_tier_k, random_state=SEED, n_init=10)
raw_tier_labels = tier_gmm.fit_predict(confidence_col)

# Order components by mean confidence (low -> high) so labels map correctly regardless
# of which internal component index GMM happened to assign each group.
component_order = np.argsort(tier_gmm.means_.flatten())
if optimal_tier_k == 3:
    tier_names = {component_order[0]: 'Low', component_order[1]: 'Medium', component_order[2]: 'High'}
else:
    tier_names = {component_order[0]: 'Low', component_order[1]: 'High'}

confidence_tiers = np.array([tier_names[c] for c in raw_tier_labels], dtype=object)

n_overridden = int(is_archetype_outlier.sum() - (confidence_tiers[is_archetype_outlier] == 'Low').sum())
confidence_tiers[is_archetype_outlier] = 'Low'

print(f'\nNatural tier sizes (pre-override): {pd.Series([tier_names[c] for c in raw_tier_labels]).value_counts().to_dict()}')
print(f'Patients pulled into Low tier by the outlier override: {n_overridden}')

scored_confidence = high_benefit_test.reset_index(drop=True).copy()
scored_confidence['gmm_archetype'] = test_gmm_labels
scored_confidence['gmm_max_posterior'] = test_gmm_max_posterior
scored_confidence['typicality_score'] = typicality_score
scored_confidence['knn_similarity'] = knn_similarity
scored_confidence['clinical_confidence_score'] = confidence_score_normalized
scored_confidence['confidence_tier'] = confidence_tiers
scored_confidence['is_archetype_outlier'] = is_archetype_outlier
scored_confidence['hdbscan_noise_flag'] = test_noise_flag

tier_order_present = [t for t in ['High', 'Medium', 'Low'] if t in set(confidence_tiers)]
tier_summary = scored_confidence.groupby('confidence_tier').agg(
    n=('clinical_confidence_score', 'count'),
    avg_confidence=('clinical_confidence_score', 'mean'),
    avg_benefit=('benefit_score', 'mean'),
    avg_posterior=('gmm_max_posterior', 'mean'),
    avg_typicality=('typicality_score', 'mean'),
    n_outliers=('is_archetype_outlier', 'sum'),
    n_hdbscan_noise=('hdbscan_noise_flag', 'sum'),
).reindex(tier_order_present).reset_index()

display(tier_summary)
tier_summary.to_csv(OUTPUT_DIR / 'confidence_tier_summary.csv', index=False)
scored_confidence.to_csv(OUTPUT_DIR / 'scored_confidence_test_patients.csv', index=False)

In [13]:
# Cross-tabulation of HDBSCAN vs Confidence Tier
overlap_table = pd.crosstab(
    scored_confidence["hdbscan_noise_flag"],
    scored_confidence["confidence_tier"],
    margins=True
)

display(overlap_table)

# --- Deliverable schema: print literal column list (P0-4) ---
print(f'\nscored_confidence has {len(scored_confidence.columns)} columns:')
print(list(scored_confidence.columns))
pd.DataFrame({'column': list(scored_confidence.columns)}).to_csv(
    OUTPUT_DIR / 'clinical_confidence_scored_schema.csv', index=False)

# ============================================================================
# Ground-truth validation by tier (synthetic true_benefit)
# The real test: does DR benefit_score track true_benefit BETTER inside the High tier
# than the Low tier? If yes, the confidence score is doing what it claims.
# ============================================================================
from scipy import stats as _spstats_tb
_sc = scored_confidence.copy()
if 'true_benefit' not in _sc.columns and 'true_benefit' in scored_test.columns and 'member_id' in _sc.columns:
    _sc = _sc.merge(scored_test[['member_id', 'true_benefit']], on='member_id', how='left')

if 'true_benefit' in _sc.columns and _sc['true_benefit'].notna().any():
    rows = []
    for tier, g in _sc.groupby('confidence_tier'):
        if len(g) > 2 and g['true_benefit'].notna().sum() > 2:
            sp = _spstats_tb.spearmanr(g['benefit_score'], g['true_benefit'])
            sp_rho, sp_p = float(sp.statistic), float(sp.pvalue)
        else:
            sp_rho, sp_p = np.nan, np.nan
        rows.append({'confidence_tier': tier, 'n': len(g),
                     'mean_true_benefit': float(g['true_benefit'].mean()),
                     'std_true_benefit': float(g['true_benefit'].std()),
                     'spearman_benefit_vs_true': sp_rho, 'spearman_p': sp_p})
    order = [t for t in ['High', 'Medium', 'Low'] if t in set(_sc['confidence_tier'])]
    true_benefit_by_tier = (pd.DataFrame(rows).set_index('confidence_tier')
                            .reindex(order).reset_index())
    true_benefit_by_tier.to_csv(OUTPUT_DIR / 'clinical_confidence_true_benefit_by_tier.csv', index=False)
    print('\nGround-truth validation by confidence tier:')
    display(true_benefit_by_tier)
else:
    print('\ntrue_benefit not available in DR test output — skipping ground-truth validation.')

---
## Step 9 – Visualization: Archetypes in PCA Space

3D scatter of training patients colored by archetype, with GMM centroids and covariance ellipsoids. Falls back to 2D if fewer than 3 PCA components were retained.
---

In [14]:
import sys
print(sys.executable)

In [15]:
import sys
!{sys.executable} -m pip show kaleido

In [16]:
import plotly
import kaleido
import platform
import shutil
import importlib.metadata as metadata

print("Plotly:", plotly.__version__)
print("Kaleido:", metadata.version("kaleido"))
print("Platform:", platform.platform())

print("Chrome:", shutil.which("google-chrome"))
print("Chromium:", shutil.which("chromium"))
print("Chromium-browser:", shutil.which("chromium-browser"))

In [17]:
import subprocess

chrome = "/home/sagemaker-user/.local/share/choreographer/deps/chrome-linux64/chrome"

result = subprocess.run(
    [chrome, "--version"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

In [18]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from matplotlib import cm, colors

# ----------------------------
# Create dataframe
# ----------------------------
plot_df = pd.DataFrame({
    "PC1": X_train_pca[:, 0],
    "PC2": X_train_pca[:, 1],
    "PC3": X_train_pca[:, 2],
    "Archetype": gmm_labels.astype(str)
})

# ----------------------------
# Evenly spaced Viridis colors
# ----------------------------
viridis = cm.get_cmap("viridis", optimal_k)

color_map = {
    str(i): colors.to_hex(viridis(i))
    for i in range(optimal_k)
}

# ----------------------------
# Interactive scatter
# ----------------------------
fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Archetype",
    color_discrete_map=color_map,
    opacity=0.7,
    title=f"GMM Archetypes (PC1-3 variance: {pca_cluster.explained_variance_ratio_[:3].sum():.1%})"
)
fig.update_traces(marker=dict(size=3), selector=dict(mode='markers'))

from matplotlib import cm, colors

# Viridis colors matching the clusters
viridis = cm.get_cmap("viridis", optimal_k)

for i in range(optimal_k):
    fig.add_trace(
        go.Scatter3d(
            x=[gmm.means_[i, 0]],
            y=[gmm.means_[i, 1]],
            z=[gmm.means_[i, 2]],
            mode="markers+text",
            marker=dict(
    symbol="diamond",
    size=14,
    color=colors.to_hex(viridis(i)),
    line=dict(
        color="red",
        width=8
    )
),
            text=[f"C{i}"],
            textposition="top center",
            name=f"Centroid {i}"
        )
    )

# ----------------------------
# Axis labels
# ----------------------------
fig.update_layout(
    scene=dict(
        xaxis_title=f"PC1 ({pca_cluster.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca_cluster.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca_cluster.explained_variance_ratio_[2]:.1%})",
    ),
    legend_title="Archetype",
    width=900,
    height=750
)

fig.show()

# Export 3D archetype plot: static PNG for README + interactive HTML for drill-down
fig.write_image(OUTPUT_DIR / 'archetype_scatter_3d.png', width=900, height=750, scale=2)
fig.write_html(OUTPUT_DIR / 'archetype_scatter_3d.html', include_plotlyjs='cdn')
print(f'Saved: archetype_scatter_3d.png + archetype_scatter_3d.html')

# --- 2D Archetype scatter (PC1 vs PC2) — viridis palette with centroid markers ---
import plotly.graph_objects as go
from matplotlib.colors import to_hex
import matplotlib.cm as cm

unique_archetypes = sorted(set(gmm_labels))
viridis = cm.get_cmap('viridis', len(unique_archetypes))

fig2d = go.Figure()
for i, arch in enumerate(unique_archetypes):
    mask = gmm_labels == arch
    fig2d.add_trace(go.Scatter(
        x=X_train_pca[mask, 0], y=X_train_pca[mask, 1],
        mode='markers', name=f'Archetype {arch}',
        marker=dict(color=to_hex(viridis(i)), size=8, opacity=0.7),
    ))

# Centroids as large X markers
centroids_pca = np.array([X_train_pca[gmm_labels == a].mean(axis=0) for a in unique_archetypes])
for i, arch in enumerate(unique_archetypes):
    fig2d.add_trace(go.Scatter(
        x=[centroids_pca[i, 0]], y=[centroids_pca[i, 1]],
        mode='markers', name=f'Centroid {arch}',
        marker=dict(color=to_hex(viridis(i)), size=18, symbol='x', line=dict(width=3, color='black')),
        showlegend=False,
    ))

fig2d.update_layout(
    title=f'GMM Archetypes (PC1 vs PC2, {pca_cluster.explained_variance_ratio_[:2].sum():.1%} variance)',
    xaxis_title=f'PC1 ({pca_cluster.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca_cluster.explained_variance_ratio_[1]:.1%})',
    width=800, height=600,
)
fig2d.show()
fig2d.write_image(OUTPUT_DIR / 'archetype_scatter_2d.png', width=800, height=600, scale=2)
fig2d.write_html(OUTPUT_DIR / 'archetype_scatter_2d.html', include_plotlyjs='cdn')
print(f'Saved: archetype_scatter_2d.png + archetype_scatter_2d.html')

---
## Step 9b – Visualization: Confidence Tiers, With Outlier Overlay

Test patients colored by confidence tier (green=High, amber=Medium, red=Low). Patients flagged as archetype outliers (Step 6) get a black ring marker overlay, so the plot distinguishes 'Low because ambiguous between archetypes' from 'Low because it doesn't resemble the training population at all.'
---

In [19]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ----------------------------------
# Data for plotting
# ----------------------------------
plot_df = pd.DataFrame({
    "PC1": X_test_pca[:, 0],
    "PC2": X_test_pca[:, 1],
    "PC3": X_test_pca[:, 2],
    "Confidence Tier": scored_confidence["confidence_tier"],
    "Confidence Score": scored_confidence["clinical_confidence_score"],
    "Posterior": scored_confidence["gmm_max_posterior"],
    "Typicality": scored_confidence["typicality_score"],
    "Archetype": scored_confidence["gmm_archetype"].astype(str)
})

# ----------------------------------
# Colors
# ----------------------------------
tier_colors = {
    "High": "#2ca02c",
    "Medium": "#ff9f1c",
    "Low": "#d62728"
}

# ----------------------------------
# Interactive scatter
# ----------------------------------
fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Confidence Tier",
    color_discrete_map=tier_colors,
    hover_data={
        "Confidence Score": ":.3f",
        "Posterior": ":.3f",
        "Typicality": ":.3f",
        "Archetype": True,
        "PC1": False,
        "PC2": False,
        "PC3": False,
    },
    opacity=0.8,
    title="Test Patients by Clinical Confidence Tier"
)
fig.update_traces(marker=dict(size=3), selector=dict(mode='markers'))

# ----------------------------------
# Add GMM centroids
# ----------------------------------
fig.add_trace(
    go.Scatter3d(
        x=gmm.means_[:, 0],
        y=gmm.means_[:, 1],
        z=gmm.means_[:, 2],
        mode="markers+text",
        text=[f"A{i}" for i in range(optimal_k)],
        textposition="top center",
        marker=dict(
            symbol="diamond",
            size=12,
            color="black",
            line=dict(color="white", width=3)
        ),
        name="GMM Archetypes"
    )
)

# ----------------------------------
# Axis labels
# ----------------------------------
fig.update_layout(
    scene=dict(
        xaxis_title=f"PC1 ({pca_cluster.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca_cluster.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca_cluster.explained_variance_ratio_[2]:.1%})"
    ),
    width=950,
    height=800,
    legend_title="Confidence Tier"
)

fig.show()

# Export 3D confidence tier plot: static PNG for README + interactive HTML
fig.write_image(OUTPUT_DIR / 'confidence_tiers_3d.png', width=950, height=800, scale=2)
fig.write_html(OUTPUT_DIR / 'confidence_tiers_3d.html', include_plotlyjs='cdn')
print(f'Saved: confidence_tiers_3d.png + confidence_tiers_3d.html')

# --- 2D Confidence Tier scatter (PC1 vs PC2) — red/green with centroid X markers ---
tier_color_map = {'High': 'green', 'Medium': 'orange', 'Low': 'red'}
unique_tiers = [t for t in ['High', 'Medium', 'Low'] if t in set(confidence_tiers)]

fig2d_tier = go.Figure()
for tier in unique_tiers:
    mask = confidence_tiers == tier
    fig2d_tier.add_trace(go.Scatter(
        x=X_test_pca[mask, 0], y=X_test_pca[mask, 1],
        mode='markers', name=tier,
        marker=dict(color=tier_color_map[tier], size=9, opacity=0.8),
    ))

# Centroid X markers for GMM archetypes (not per-tier)
centroids_pca_test = np.array([X_train_pca[gmm_labels == a].mean(axis=0) for a in unique_archetypes])
for i, arch in enumerate(unique_archetypes):
    fig2d_tier.add_trace(go.Scatter(
        x=[centroids_pca_test[i, 0]], y=[centroids_pca_test[i, 1]],
        mode='markers', name=f'Archetype {arch} centroid',
        marker=dict(color='black', size=20, symbol='x', line=dict(width=3, color='black')),
        showlegend=True,
    ))

fig2d_tier.update_layout(
    title=f'Confidence Tiers (PC1 vs PC2, {pca_cluster.explained_variance_ratio_[:2].sum():.1%} variance)',
    xaxis_title=f'PC1 ({pca_cluster.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca_cluster.explained_variance_ratio_[1]:.1%})',
    width=800, height=600,
)
fig2d_tier.show()
fig2d_tier.write_image(OUTPUT_DIR / 'confidence_tiers_2d.png', width=800, height=600, scale=2)
fig2d_tier.write_html(OUTPUT_DIR / 'confidence_tiers_2d.html', include_plotlyjs='cdn')
print(f'Saved: confidence_tiers_2d.png + confidence_tiers_2d.html')

---
## Output Index
---

In [20]:
output_files = sorted(OUTPUT_DIR.iterdir())
print(f'Outputs saved to: {OUTPUT_DIR}\n')
for f in output_files:
    print(f'  {f.name}')

print('\n--- Summary ---')
print(f'Training reference: {len(high_benefit_train)} high-benefit patients')
print(f'Test scored: {len(high_benefit_test)} high-benefit patients')
print(f'Clustering features: {len(CLUSTER_FEATURES)} (post correlation pruning)')
print(f'PCA components: {pca_cluster.n_components_} (target variance {PCA_VARIANCE_TARGET:.0%})')
print(f'GMM: k={optimal_k}, covariance_type={GMM_COVARIANCE_TYPE}, reg_covar={GMM_REG_COVAR}')
print(f'Stability: {stability_assessment} (mean bootstrap ARI={bootstrap_ari.mean():.3f})')
print(f'Archetype outliers flagged: {is_archetype_outlier.sum()} / {len(is_archetype_outlier)} '
      f'(HDBSCAN-corroborated: {(is_archetype_outlier & test_noise_flag).sum()})')